# <center><span style="color:#336699">
SER347/CAP419 - Introdução a Programação com Dados Geoespaciais</span></center>
<hr style="border:2px solid #0077b9;">

<br/>

<div style="text-align: center;font-size: 200%;">
   Registro de imagens MUX L2 com base no Sentinel-2 <br/>
</div>


<br/>

<div style="text-align: center;font-size: 90%;">
    Carla Almeida e Tayná Florentino <sup><a href=""></i></a></sup>
    <br/><br/>
     Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Atualização: 23 de Maio de 2026
</div>

<br/>

<div style="text-align: justify;  margin-left: 25%; margin-right: 25%;">
    <b>Resumo.</b> O catálogo de imagens do INPE oferece um conjunto de imagens atualizado diariamente. O sensor MUX está disponível em 2 satélites brasileiros (CBERS-4 e CBERS-4A). Algumas imagens possuem nível de processamento L2, o que indica que não passaram por todas as etapas de registro geométrico. Entretanto, é possível efetuar o registro utilizando como base uma outra imagem, de resolução espacial semelhante, obita em uma época parecida.Dentre os produtos de imagens oferecidos pelo Dataspace Copernicus, há o dado SENTINEL_2_L1C em um composto com baixa quantidade de nuvens, chamado LEAST_CC. Esse produto pode ser uma referência para melhorar o registro das imagens L2. Dada uma cena do sensor MUX disponível no catálogo do INPE, em nível de processamento L2, encontrar no Dataspace Copernicus uma cena correspondente, com as mesmas bandas disponíveis no MUX, e aplicar uma técnica de co-registro entre as imagens (usando o pacote arosics).. 
</div>

<br/>

    

### <span style="color:#336699">Satélite CBERS_4A </span>
<hr style="border:1px solid #0077b9;">

Possui três camerás para geracao de imagens. Uma delas é a  câmera Multiespectral MUX  produz imagens com resolução espacial de 16,5 metros.

* Obs: O tempo de revisita  das câmeras WPM e MUX é de 31 dias. 

Esse satélite possui as seguintes bandas  espectrais:

* Banda 5 (azul): banda do canal multiespectral da faixa do azul, com 16 metros de resolução espacial.
* Banda 6 (verde): banda do canal multiespectral da faixa do verde, com 16 metros de resolução espacial.
* Banda 7 (vermelho): banda do canal multiespectral da faixa do vermelho, com 16 metros de resolução espacial.
* Banda 8 (NIR): banda do canal multiespectral da faixa do infravermelho próximo, com 16 metros de resolução espacial.

### <span style="color:#336699">1. Verificando e Carregando o CBERS-4A MUX </span>
<hr style="border:1px solid #0077b9;">


In [ ]:
# Importando as classes da API do QGIS (PyQGIS) para controle do projeto e leitura de imagens raster

# Importando as classes da API do QGIS (PyQGIS) para controle do projeto e leitura de imagens raster
import os
from qgis.core import QgsProject, QgsRasterLayer

# Acessando a pasta onde estão as imagens
pasta = "/Users/carlaalmeida/Documents/python_qgis/Projeto/CBERS4A"

# Filtrando os arquivos .tif e adicionando uma camada no QGIS
for f in os.listdir(pasta):

    # Converte o nome do arquivo para minúsculas e checa se ele termina com '.tif'
    if f.lower().endswith('.tif'):
        caminho_completo = os.path.join(pasta, f)
        camada = QgsRasterLayer(caminho_completo, f)

        # Envia a camada de imagem que criamos DIRETO para a interface do QGIS se for válida
        if camada.isValid():
            QgsProject.instance().addMapLayer(camada)
            print(f"Camada CBERS carregada com sucesso: {f}")
        else:
            print(f"Erro ao carregar a imagem: {f}")

<div align="center">
  <img src="conf/camadas_carregadas.png" width="600"/>
</div>

### <span style="color:#336699">1.1 Visualizando as Projeções </span>
<hr style="border:1px solid #0077b9;">

    

In [ ]:
from qgis.core import QgsProject, QgsMapLayerType

# Percorre todas as camadas que estão atualmente carregadas no projeto do QGIS
for camada in QgsProject.instance().mapLayers().values():
    
# Verificando se a camada atual é do tipo Raster usando o enumerador correto
    if camada.type() == QgsMapLayerType.RasterLayer:
        
# Exiba na tela o nome da imagem e o seu código de projeção (EPSG)
        print(f"Imagem: {camada.name()} | SRC: {camada.crs().authid()}")

<div align="center">
  <img src="conf/projecao_cbers.png" width="600"/>
</div>

### <span style="color:#336699">1.2 Verificando a Projeção UTM no Sentinel 2</span>
<hr style="border:1px solid #0077b9;">

In [ ]:
import os
from qgis.core import QgsProject, QgsRasterLayer

# Adicionando o caminho onde está localizado o arquivo do Sentinel-2
pasta_copernicus = "/Users/carlaalmeida/Documents/python_qgis/Projeto/Copernicus"

# Percorre a pasta e adiciona os rasters válidos (.tif ou .jp2) ao mapa
for f in os.listdir(pasta_copernicus):
    if f.lower().endswith(('.tif', '.jp2')):
        caminho = os.path.join(pasta_copernicus, f)
        camada = QgsRasterLayer(caminho, f)
        
        if camada.isValid():
            QgsProject.instance().addMapLayer(camada)
            print(f"Camada Sentinel carregada com sucesso: {f}")

<div align="center")
  <p><b> Sobreposição espacial entre a cena do satélite Sentinel-2 e o recorte do CBERS-4A MUX na interface do QGIS.</p>
  <img src="conf/sobreposicao_img.png" width="500"/>
</div>

### <span style="color:#336699">1.3 Visualizando todas as Projeções Combinadas</span>
<hr style="border:1px solid #0077b9;">

In [ ]:
from qgis.core import QgsProject, QgsMapLayerType

# Percorre as camadas raster do projeto para validar o SRC
for camada in QgsProject.instance().mapLayers().values():
    if camada.type() == QgsMapLayerType.RasterLayer:
        crs = camada.crs()
        
        # Identifica se é Geográfico (Graus) ou Projetado (Metros/UTM)
        tipo = "Geográfico (Graus)" if crs.isGeographic() else "Projetado/UTM (Metros)"

        # Exibe o resultado formatado na tela       
        print(f"Imagem: {camada.name()}")
        print(f"  > Código EPSG: {crs.authid()}")
        print(f"  > Nome do SRC: {crs.description()}")
        print(f"  > Unidade:     {tipo}")
        print("-" * 30)

<div align="center">
  <img src="conf/projecao_final.png" width="600"/>
</div>

### <span style="color:#336699">Referências Bibliográficas</span>
<hr style="border:1px solid #0077b9;">

- INPE - Instituto Nacional de Pesquisas Espaciais. Catálogo de Imagens do Satélite CBERS-4A. Dados obtidos em Maio de 2026. Disponível em: https://www.dgi.inpe.br/.

- Copernicus Browser - Sentinel 2 .  Disponível em: https://browser.dataspace.copernicus

- Dados de Satélites   Disponível em: https://data.inpe.br/dados/satelite-cbers-4a/